# Real vs Fake Faces

We are going to `Fake-Vs-Real-Faces`. The Dataset contains two types of image which are: `Fake faces` image collected from the website [thispersondoesnotexist.com](thispersondoesnotexist.com). , `Real faces` images are re collected through the API of the website [Unsplash](https://unsplash.com/) and then the faces are cropped out of using OpenCV library.

Here some additional information about the dataset:
- Total number of images: 1288
- Number of "Fake" faces: 700
- Number of "Real" faces: 589

### 1. Import Libraries 

In [ ]:
import torch 
import torch.nn as nn
from torch.nn import Linear, Dropout, ReLU
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, random_split, Dataset
from torchvision.datasets import ImageFolder # Stream data from images stored in folders
import torch.optim as optim
import torch.nn.functional as F
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score
from torchinfo import summary

from sklearn.model_selection import StratifiedKFold, train_test_split
import os # allow s to access files
import numpy as np
from PIL import Image # Allows us to Load Images
from collections import Counter # Utility function to give us the counts of unique items in an iterable
import matplotlib.pyplot as plt
import shutil

## 2. Reconstruct the Folder structure of the Data

In [2]:
torch.manual_seed(42)

Before loading, let's reconstruct the data folder structure to give us comfort when using the `ImageFolder` class

In [3]:
def copy_image(des_dir, split_name, cls, file_list, cls_path):

    """
    des_dir: data_split
    split_name: can be "train" or "test"
    cls and cls_path: is class which can be "fake" or "real"
    file_list: list of images files

    save a copy of images from folder 'cls_path/img' into folder called 'des_dir/split_name/cls/img' 
    """
    
    for img in file_list:
        src = os.path.join(cls_path, img)
        dest = os.path.join(des_dir, split_name, cls, img)
        shutil.copy2(src, dest)

In [4]:

def split_and_save_to_disk(src_dir, des_dir):

    """
    reconstruct the folder structure 
    from:
        data/
            fake/
            real

    to:
        data_split/
            train/
                fake/
                real/
            test/
                fake/
                real/
    
    """
    
    # # splite ratios
    train_ratio = 0.8
    test_ratio = 0.2

    if os.path.exists(des_dir):
        print(f"{src_dir} already exists")
    else:
        print("reconstruct the data folder structure")

        # create folder 
        for split in ['train', 'test']:
            for cls in os.listdir(src_dir):
                os.makedirs(os.path.join(des_dir, split, cls), exist_ok=True)  # make it similar to this (des_dir/train/fake) and  (des_dir/train/real)


        # Process each class folder 
        for cls in os.listdir(src_dir):
            cls_path = os.path.join(src_dir, cls)
            if not os.path.isdir(cls_path):
                continue


            images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            # split train and (test and val)
            train_data, test_data = train_test_split(images, test_size=test_ratio, random_state=42)


            # splitting the data 
            print(f"splitting class '{cls}':\n train:{len(train_data)}\n test:{len(test_data)}")

            # des_dir, split_name, cls, file_list, cls_path
            copy_image(des_dir, "train", cls, train_data, cls_path)
            copy_image(des_dir, "test", cls, test_data, cls_path)
            
        print("Copying proces is finished")
        

## 3. Load and Prepocess the data

First Let's define the transforms pipeline for `data augmentatin` and `preprocessing` steps

In [5]:
split_and_save_to_disk("data", "data_split")

data already exists


In [6]:
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# for data preprocessing and data augmentation for trianing data
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


test_valid_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),

])

Let's load our images using `ImageFolder`. This class inherits from `DatasetFolder`  class. So, we can override the classes to create custom datasets as well. Because we need to custome dataset class becasue `ImageFolder` apply a single tansform on the whole dataset while we need to apply custome `transform` on each subset in the dataset.

In [7]:
path_to_train = './split_data/train'
path_to_test = './split_data/test'

train_data = ImageFolder(root=path_to_train, transform=None)
test_data = ImageFolder(root=path_to_test, transform=test_valid_transforms)

class CustomSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        
        actual_idx = self.subset.indices[idx]
        
        image_path, label = self.subset.dataset.samples[actual_idx]
        image = self.subset.dataset.loader(image_path) 
        
        
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        
        return image, label
    
    def __len__(self):
        return len(self.subset)
        
# Create the DataLoders
BATCH_SIZE = 16
# train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

## 3. Build Helper functions

Before starting building the model we will build a helper function to save our models

In [8]:
def save_model(epochs, model, optimizer, loss_fun, model_name):

    """
    Function to save the trained model to disk
    """
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_fun,
    }, f'outputs/{model_name}.pth')

The next function is for saving the loss and accuracy plots after the training completes

In [9]:
def save_plots(train_acc, valid_acc, test_acc, train_loss, valid_loss, test_loss):

    """
    Function to save the loss and accurcy plots to disk
    """

    # accuracy plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_acc, color='green', linestyle='-', label='train accuracy')
    plt.plot(valid_acc, color='red', linestyle='-', label='validation accuracy')
    plt.plot(test_acc, color='cyan', linestyle='-', label='test accuracy')
    plt.xlabel('EPOCHs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('outputs/accuracy.png')

    # loss plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, color='orange', linestyle='-', label='train loss')
    plt.plot(valid_loss, color='blue', linestyle='-', label='valid loss')
    plt.plot(test_loss, color='magenta', linestyle='-', label='test loss')
    plt.xlabel('EPOCHs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('outputs/loss.png')

## 4. Prepparing the Data Loaders and Stratified KFold

Now we will build our `data loader` and `stratified KFold` spliter for that we need for training.

In [10]:
N_SPLITs = 5
SEED = 42

In [11]:
def train(model, training_data, X, y, optimizer, loss_fn, Epochs = 10, N_SPLITs = 4):

    """
    Train the given model
    """


    
    skf = StratifiedKFold(n_splits=N_SPLITs, shuffle=True, random_state=SEED)
    stratified_scores = []

    # folds results 
    all_fold_results = []

    # Lists to collect metrics
    train_acc_list, valid_acc_list = [], []
    train_loss_list, valid_loss_list = [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):



        # Initialize fold-specific metrics
        fold_train_loss = []
        fold_val_loss = []
        fold_train_acc = []
        fold_val_acc = []
        
        # Create the Data Loaders
        train_subset = Subset(training_data, train_idx)
        val_subset = Subset(training_data, valid_idx)


        # Apply transforms dynamically
        train_data_fold = CustomSubset(train_subset, transform=train_transforms)
        val_data_fold = CustomSubset(val_subset, transform=test_valid_transforms)
    
        # Dataloaders
        train_loader = DataLoader(train_data_fold, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(val_data_fold, batch_size=BATCH_SIZE, shuffle=False)

        ### Training ###
        model.train() 
        print("Training")
        for epoch in range(Epochs):

            train_loss, correct, total = 0.0, 0, 0
            
            for images, labels in train_loader:
                predictions = model(images)
                labels = labels.unsqueeze(1).float()
                loss = loss_fn(predictions, labels)
                loss.backward() # backpagation

                train_loss += loss.item()
                
                optimizer.step() # update the weight
                optimizer.zero_grad() # set the gradient to zero
                
                preds = (torch.sigmoid(predictions) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)


            # calculate the accuracy and loss for the current epoch
            train_accuracy = correct / total
            epoch_train_loss = train_loss / len(train_loader)


            ### Validation ###
            model.eval()
            print("Evaluting")
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in valid_loader:
                    predictions = model(images)
                    labels = labels.unsqueeze(1).float()
                    val_loss += loss_fn(predictions, labels).item()
                    
                    preds = (torch.sigmoid(predictions) > 0.5).float()
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

                    
            val_accuracy = correct / total
            epoch_val_loss = val_loss / len(valid_loader)
     

        
            fold_train_loss.append(train_loss)
            fold_val_loss.append(val_loss)
            fold_train_acc.append(train_accuracy)
            fold_val_acc.append(val_accuracy)
            
            print(f"Epoch {epoch+1}/{Epochs} - Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

            
        # Store metrics for this fold
        all_fold_results.append({
            'fold': fold + 1,
            'train_loss': fold_train_loss,
            'val_loss': fold_val_loss,
            'train_acc': fold_train_acc,
            'val_acc': fold_val_acc
        })


    avg_val_acc = sum(r['val_acc'][-1] for r in all_fold_results) / N_SPLITs
    avg_val_loss = sum(r['val_loss'][-1] for r in all_fold_results) / N_SPLITs
    print(f"\nAverage Validation Accuracy across folds: {avg_val_acc:.4f}")
    print(f"Average Validation Loss across folds: {avg_val_loss:.4f}")

    return all_fold_results
        
              
    

## 4, 5. Build and Train the Models

> ### 4.1 AlexNets

In [38]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import alexnet
from torchvision.models.alexnet import AlexNet_Weights

# initialize the pretrained alexnet
alexnet = alexnet(weights=AlexNet_Weights.DEFAULT)

alexnet.eval()

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

> ### 4.2 Freeze early layers 

In [39]:
for parameter in alexnet.features.parameters():
    parameter.requires_grad = False


> ### 4.3 Modfiy the AlexNet (output layer)

In [40]:
alexnet.classifier = nn.Sequential(
    Dropout(p=0.5, inplace=False),
    Linear(in_features=9216, out_features=4096, bias=True),
    ReLU(inplace=True),
    Dropout(p=0.5, inplace=False),
    Linear(in_features=4096, out_features=4096, bias=True),
    ReLU(inplace=True),
    Linear(in_features=4096, out_features=1, bias=True),
)

> ### 4.4 Get model summary

In [41]:
summary(alexnet,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
AlexNet (AlexNet)                        [16, 3, 224, 224]  [16, 1]            --                 Partial
├─Sequential (features)                  [16, 3, 224, 224]  [16, 256, 6, 6]    --                 False
│    └─Conv2d (0)                        [16, 3, 224, 224]  [16, 64, 55, 55]   (23,296)           False
│    └─ReLU (1)                          [16, 64, 55, 55]   [16, 64, 55, 55]   --                 --
│    └─MaxPool2d (2)                     [16, 64, 55, 55]   [16, 64, 27, 27]   --                 --
│    └─Conv2d (3)                        [16, 64, 27, 27]   [16, 192, 27, 27]  (307,392)          False
│    └─ReLU (4)                          [16, 192, 27, 27]  [16, 192, 27, 27]  --                 --
│    └─MaxPool2d (5)                     [16, 192, 27, 27]  [16, 192, 13, 13]  --                 --
│    └─Conv2d (6)                        [16, 192, 13, 13]  [16, 384, 

> ### 4.4 Train the Model

In [42]:
# Specifiy learing rate and number of Epochs
Alex_LR = 0.001
alex_Epochs = 2
alex_n_splits = 4
# Define loss and optimizer
alex_loss_fun = nn.BCEWithLogitsLoss()
alex_optimizer = torch.optim.Adam(alexnet.parameters(), lr=Alex_LR)


train_X = [train_data.samples[img][0] for img in range(len(train_data.samples))]
train_y = [train_data.samples[label][1] for label in range(len(train_data.samples))]

skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
BATCH_SIZE = 16

In [43]:
result = train(alexnet, train_data, train_X, train_y, alex_optimizer, alex_loss_fun, alex_Epochs, N_SPLITs = alex_n_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.5871, Train Acc: 0.9288, Val Loss: 0.0607, Val Acc: 0.9806
Evaluting
Epoch 2/2 - Train Loss: 0.0345, Train Acc: 0.9845, Val Loss: 0.0431, Val Acc: 0.9922
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0276, Train Acc: 0.9884, Val Loss: 0.0189, Val Acc: 0.9922
Evaluting
Epoch 2/2 - Train Loss: 0.0003, Train Acc: 1.0000, Val Loss: 0.0000, Val Acc: 1.0000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0641, Train Acc: 0.9884, Val Loss: 0.0020, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.0141, Train Acc: 0.9961, Val Loss: 0.0000, Val Acc: 1.0000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1118, Train Acc: 0.9806, Val Loss: 0.1358, Val Acc: 0.9572
Evaluting
Epoch 2/2 - Train Loss: 0.0392, Train Acc: 0.9948, Val Loss: 0.0000, Val Acc: 1.0000

Average Validation Accuracy across folds: 0.9981
Average Validation Loss across folds: 0.1835


> ### 5.1 VGG16

In [30]:
# Load the VGG16 model and weights 
from torchvision.models import vgg16, VGG16_Weights

# initialize the pretrained alexnet
vgg16_model = vgg16(weights=VGG16_Weights.DEFAULT)

vgg16_model.eval()

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

> ### 5.2 Freeze early layers 

In [33]:
for parameter in vgg16_model.features.parameters():
    parameter.requires_grad = False

> ### 5.3 Get model summary

In [34]:
summary(vgg16_model,
        input_size=(16, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
VGG (VGG)                                [16, 3, 224, 224]  [16, 1000]         --                 Partial
├─Sequential (features)                  [16, 3, 224, 224]  [16, 512, 7, 7]    --                 False
│    └─Conv2d (0)                        [16, 3, 224, 224]  [16, 64, 224, 224] (1,792)            False
│    └─ReLU (1)                          [16, 64, 224, 224] [16, 64, 224, 224] --                 --
│    └─Conv2d (2)                        [16, 64, 224, 224] [16, 64, 224, 224] (36,928)           False
│    └─ReLU (3)                          [16, 64, 224, 224] [16, 64, 224, 224] --                 --
│    └─MaxPool2d (4)                     [16, 64, 224, 224] [16, 64, 112, 112] --                 --
│    └─Conv2d (5)                        [16, 64, 112, 112] [16, 128, 112, 112] (73,856)           False
│    └─ReLU (6)                          [16, 128, 112, 112] [16, 

> ### 5.4 Modfiy the VGG16 (output layer)

In [35]:
vgg16_model.classifier = nn.Sequential(
    Linear(in_features=25088, out_features=4096, bias=True),
    ReLU(inplace=True),
    Dropout(p=0.5, inplace=False),
    Linear(in_features=4096, out_features=4096, bias=True),
    ReLU(inplace=True),
    Dropout(p=0.5, inplace=False),
    Linear(in_features=4096, out_features=1, bias=True)
)

> ### 5.5 Train the model

In [36]:
# Specifiy learing rate and number of Epochs
vgg16_LR = 0.001
vgg16_Epochs = 2
vgg16_n_splits = 4
# Define loss and optimizer
vgg16_loss_fun = nn.BCEWithLogitsLoss()
vgg16_optimizer = torch.optim.Adam(vgg16_model.parameters(), lr=vgg16_LR)


In [37]:
result_vgg16_model = train(vgg16_model, train_data, train_X, train_y, vgg16_optimizer, vgg16_loss_fun, vgg16_Epochs, vgg16_n_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.8633, Train Acc: 0.9250, Val Loss: 0.0454, Val Acc: 0.9884
Evaluting
Epoch 2/2 - Train Loss: 0.2215, Train Acc: 0.9754, Val Loss: 0.1834, Val Acc: 0.9922
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0425, Train Acc: 0.9948, Val Loss: 0.0064, Val Acc: 0.9961
Evaluting
Epoch 2/2 - Train Loss: 0.0483, Train Acc: 0.9974, Val Loss: 0.0001, Val Acc: 1.0000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0053, Train Acc: 0.9974, Val Loss: 0.0004, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.0132, Train Acc: 0.9987, Val Loss: 0.1476, Val Acc: 0.9767
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0481, Train Acc: 0.9948, Val Loss: 0.0030, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.1555, Train Acc: 0.9935, Val Loss: 0.3519, Val Acc: 0.9728

Average Validation Accuracy across folds: 0.9854
Average Validation Loss across folds: 2.9022


## 7. Test the models

In [44]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score

def test(model, test_loader, loss_fn):
    """
    Test the model and return testing metrics
    """
    model.eval()
    print("Testing")
    
    val_loss, correct, total = 0.0, 0, 0
    y_pred, y_true = [], []
    
    # Initialize metrics
    precision = BinaryPrecision()
    recall = BinaryRecall()
    f1 = BinaryF1Score()
    
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = model(images)
            labels = labels.unsqueeze(1).float()
            
   
            val_loss += loss_fn(predictions, labels).item()
            

            preds = (torch.sigmoid(predictions) > 0.5).float()
            
            y_pred.append(preds)
            y_true.append(labels)
            
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    y_pred = torch.cat(y_pred, dim=0)
    y_true = torch.cat(y_true, dim=0)
    
    # Metrics
    test_accuracy = correct / total
    test_loss = val_loss / len(test_loader)
    test_precision = precision(y_pred, y_true).item()
    test_recall = recall(y_pred, y_true).item()
    test_f1 = f1(y_pred, y_true).item()
    
    return {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1': test_f1,
    }

In [45]:
alexnet_test_results = test(alexnet, test_loader, alex_loss_fun)

Testing


In [46]:
vgg16_test_results = test(vgg16_model, test_loader, vgg16_loss_fun)

Testing


In [47]:
alexnet_test_results

{'test_loss': 0.0018265157755296799,
 'test_accuracy': 1.0,
 'test_precision': 1.0,
 'test_recall': 1.0,
 'test_f1': 1.0}

In [48]:
vgg16_test_results

{'test_loss': 0.7709746516057696,
 'test_accuracy': 0.937984496124031,
 'test_precision': 1.0,
 'test_recall': 0.8644067645072937,
 'test_f1': 0.9272727370262146}

In [49]:
avg_val_acc = sum(r['val_acc'][-1] for r in result) / alex_n_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result) / alex_n_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result) / alex_n_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result) / alex_n_splits
print(f"Alexnet average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"Alexnet average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\nAlexnet average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"alexnet average Traing Loss across folds: {avg_train_loss:.4f}")

Alexnet average Validation Accuracy across folds: 0.9981
Alexnet average Validation Loss across folds: 0.1835

Alexnet average Training Accuracy across folds: 0.9939
alexnet average Traing Loss across folds: 1.0786


In [50]:
avg_val_acc = sum(r['val_acc'][-1] for r in result_vgg16_model) / vgg16_n_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result_vgg16_model) / vgg16_n_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result_vgg16_model) / vgg16_n_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result_vgg16_model) / vgg16_n_splits
print(f"VGG16 average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"VGG16 average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\nVGG16 average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"VGG16 average Traing Loss across folds: {avg_train_loss:.4f}")

VGG16 average Validation Accuracy across folds: 0.9854
VGG16 average Validation Loss across folds: 2.9022

VGG16 average Training Accuracy across folds: 0.9913
VGG16 average Traing Loss across folds: 5.3708
